In [1]:
from datasets import load_dataset
from collections import Counter
import gc
import ast
import pandas as pd

In [2]:
full_val_df = pd.read_parquet("data/danbooru2025_val.parquet")
full_val_df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url,tags,image_path
0,3803458,2020-02-29T02:04:09.022-05:00,65,s,623996,1girl antenna_hair armpits bed_sheet blush bre...,al_azif_(demonbane),demonbane,https://cdn.donmai.us/360x360/57/a8/57a82bf4a1...,"[1girl, antenna_hair, armpits, bed_sheet, blus...",data/images/danbooru2025_val/3803458.jpg
1,7615873,2024-05-08T16:52:36.138-04:00,106,e,126497,1boy 1girl all_fours anal ass barefoot black_t...,megumin satou_kazuma,kono_subarashii_sekai_ni_shukufuku_wo!,https://cdn.donmai.us/360x360/04/5e/045e482079...,"[1boy, 1girl, all_fours, anal, ass, barefoot, ...",data/images/danbooru2025_val/7615873.jpg
2,5807689,2022-11-08T06:41:09.289-05:00,99,s,295115,1girl alternate_hairstyle breasts cleavage clo...,arisugawa_natsuha,idolmaster idolmaster_shiny_colors,https://cdn.donmai.us/360x360/dd/58/dd589e8223...,"[1girl, alternate_hairstyle, breasts, cleavage...",data/images/danbooru2025_val/5807689.jpg
3,5771329,2022-10-24T13:40:25.390-04:00,135,e,2100081,1boy 1girl :o arm_grab bar_censor bouncing_bre...,,original,https://cdn.donmai.us/360x360/7e/db/7edb0f7795...,"[1boy, 1girl, :o, arm_grab, bar_censor, bounci...",data/images/danbooru2025_val/5771329.jpg
4,2987219,2018-01-14T16:47:46.466-05:00,62,q,154001,1girl armor black_panties blue_eyes blue_hair ...,nanami_yachiyo,magia_record:_mahou_shoujo_madoka_magica_gaide...,https://cdn.donmai.us/360x360/ff/7e/ff7edf51b7...,"[1girl, armor, black_panties, blue_eyes, blue_...",data/images/danbooru2025_val/2987219.jpg
...,...,...,...,...,...,...,...,...,...,...,...
99995,5599936,2022-08-17T22:49:56.764-04:00,53,s,577595,1girl ahoge animal_ears bare_legs black_skirt ...,,original,https://cdn.donmai.us/360x360/5b/62/5b620ccca4...,"[1girl, ahoge, animal_ears, bare_legs, black_s...",data/images/danbooru2025_val/5599936.jpg
99996,8643081,2024-12-31T10:01:59.979-05:00,143,e,9777813,1boy 1girl :q against_glass anus ass breasts c...,kita_ikuyo,bocchi_the_rock!,https://cdn.donmai.us/360x360/c6/73/c673f9a7f9...,"[1boy, 1girl, :q, against_glass, anus, ass, br...",data/images/danbooru2025_val/8643081.jpg
99997,6042767,2023-02-04T16:56:08.190-05:00,231,e,5237793,1boy 1girl ? animal_ears anus ass bar_censor b...,tsunomaki_watame,hololive,https://cdn.donmai.us/360x360/e9/3a/e93ae304c9...,"[1boy, 1girl, ?, animal_ears, anus, ass, bar_c...",data/images/danbooru2025_val/6042767.jpg
99998,5870341,2022-12-03T09:46:20.261-05:00,52,s,604539,1girl body_freckles bra bracelet breasts cleav...,beelzebub_(helltaker),helltaker,https://cdn.donmai.us/360x360/22/0b/220bd87446...,"[1girl, body_freckles, bra, bracelet, breasts,...",data/images/danbooru2025_val/5870341.jpg


In [3]:
# convert the datetime col
full_val_df["media_asset_created_at"] = pd.to_datetime(
    full_val_df["media_asset_created_at"],
    utc=True,
)

In [4]:
# get all tags from a DF
def get_tags(df_):
    all_tags = Counter()
    for tags in df_["tags"]:
        all_tags.update(tags)
    return all_tags

In [5]:
# all tags after 2024-02
new_df = full_val_df[full_val_df["media_asset_created_at"] >= pd.Timestamp("2024-03-01", tz="UTC")]
new_tags = get_tags(new_df)
len(new_tags)

10386

In [7]:
for tag in new_tags:
    no_text = True
    for c in tag:
        if c.isalpha():
            no_text = False
            break
    if no_text:
        print(f"tag: {tag}, count: {new_tags[tag]}")

tag: >_<, count: 41
tag: ?, count: 216
tag: :3, count: 178
tag: !?, count: 57
tag: !, count: 74
tag: 69, count: 14
tag: +++, count: 15
tag: @_@, count: 161
tag: :>, count: 18
tag: :>=, count: 36
tag: :/, count: 44
tag: ^_^, count: 70
tag: :|, count: 13
tag: =_=, count: 15
tag: ..., count: 91
tag: ^^^, count: 87
tag: !!, count: 29
tag: 3:, count: 13
tag: <|>_<|>, count: 6
tag: +_+, count: 32
tag: :<, count: 31
tag: >:), count: 10
tag: ...?, count: 1
tag: >:(, count: 7
tag: 2024, count: 14
tag: \||/, count: 15
tag: ;), count: 29
tag: ??, count: 31
tag: 2025, count: 13
tag: |_|, count: 4
tag: ;3, count: 3
tag: 0_0, count: 2
tag: 2023, count: 2


In [8]:
# most popular tags
stags = sorted((-c, t) for t, c in new_tags.items())
stags = [t.replace("_", " ") for c, t in stags][:1000]
", ".join(stags)


"1girl, breasts, long hair, looking at viewer, solo, blush, large breasts, smile, navel, open mouth, nipples, simple background, black hair, thighs, white background, 1boy, cleavage, hair ornament, ass, animal ears, blue eyes, shirt, bare shoulders, blue archive, hetero, sweat, very long hair, hair between eyes, gloves, multicolored hair, nude, short hair, red eyes, censored, underwear, blonde hair, thighhighs, penis, halo, collarbone, closed mouth, jewelry, swimsuit, multiple girls, holding, panties, pussy, long sleeves, tail, skirt, bikini, sitting, heart, purple eyes, stomach, brown hair, medium breasts, sex, dress, huge breasts, white shirt, virtual youtuber, standing, alternate costume, blue hair, 2girls, white hair, pink hair, indoors, lying, grey hair, jacket, bow, earrings, solo focus, completely nude, yellow eyes, twintails, official alternate costume, original, choker, green eyes, sidelocks, pantyhose, small breasts, cowboy shot, hololive, tongue, ahoge, mosaic censoring, cum

In [10]:
df = new_df[:5000]
df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url,tags,image_path
1,7615873,2024-05-08 20:52:36.138000+00:00,106,e,126497,1boy 1girl all_fours anal ass barefoot black_t...,megumin satou_kazuma,kono_subarashii_sekai_ni_shukufuku_wo!,https://cdn.donmai.us/360x360/04/5e/045e482079...,"[1boy, 1girl, all_fours, anal, ass, barefoot, ...",data/images/danbooru2025_val/7615873.jpg
8,7711000,2024-06-14 02:28:49.403000+00:00,83,q,3930555,1girl back backboob backless_dress backless_ou...,yor_briar,spy_x_family,https://cdn.donmai.us/360x360/36/15/3615871497...,"[1girl, back, backboob, backless_dress, backle...",data/images/danbooru2025_val/7711000.jpg
9,7570805,2024-05-12 12:43:56.579000+00:00,65,s,264075,1girl ahoge animal_ear_fluff animal_ears back_...,hina_(blue_archive) mash_kyrielight mash_kyrie...,blue_archive fate/grand_order fate_(series),https://cdn.donmai.us/360x360/3b/bb/3bbb87d7ad...,"[1girl, ahoge, animal_ear_fluff, animal_ears, ...",data/images/danbooru2025_val/7570805.jpg
10,7291029,2024-03-04 08:48:56.042000+00:00,54,s,1006806,1girl animal_ears bare_shoulders black_leotard...,hatsune_miku,rabbit_hole_(vocaloid) vocaloid,https://cdn.donmai.us/360x360/c5/48/c548a47499...,"[1girl, animal_ears, bare_shoulders, black_leo...",data/images/danbooru2025_val/7291029.jpg
11,7841062,2024-07-11 14:15:40.595000+00:00,86,q,715030,4girls ace_trainer_(pokemon)_(cosplay) antenna...,ace_trainer_(pokemon) dawn_(pokemon) hilda_(po...,pokemon pokemon_bw pokemon_bw2 pokemon_dppt po...,https://cdn.donmai.us/360x360/22/bb/22bb2ac0e9...,"[4girls, antenna_hair, blush, bow_hairband, br...",data/images/danbooru2025_val/7841062.jpg
...,...,...,...,...,...,...,...,...,...,...,...
26469,8517246,2024-12-03 16:34:48.950000+00:00,59,g,388383,1boy 1girl blush collared_shirt crossed_arms e...,wise_(zenless_zone_zero) zhu_yuan,zenless_zone_zero,https://cdn.donmai.us/360x360/3f/b7/3fb753f1eb...,"[1boy, 1girl, blush, collared_shirt, crossed_a...",data/images/danbooru2025_val/8517246.jpg
26478,8043413,2024-08-23 21:51:03.857000+00:00,295,e,1804812,1boy 1girl artist_name blush closed_eyes erect...,alexandrina_sebastiane,zenless_zone_zero,https://cdn.donmai.us/360x360/c8/15/c815e39e2a...,"[1boy, 1girl, artist_name, blush, closed_eyes,...",data/images/danbooru2025_val/8043413.jpg
26481,8277191,2024-10-11 19:21:40.072000+00:00,167,e,375463,2girls ass barbell_piercing barefoot blue_eyes...,elizabeth_(bioshock_infinite) lara_croft,bioshock_(series) bioshock_infinite tomb_raider,https://cdn.donmai.us/360x360/0a/18/0a185c4d55...,"[2girls, ass, barbell_piercing, barefoot, blue...",data/images/danbooru2025_val/8277191.jpg
26482,8507119,2024-12-01 09:25:57.320000+00:00,101,q,2464085,1girl black_hair blunt_bangs blush bodysuit br...,sabrina_(pokemon),pokemon pokemon_frlg,https://cdn.donmai.us/360x360/dd/1b/dd1bbd6173...,"[1girl, black_hair, blunt_bangs, blush, bodysu...",data/images/danbooru2025_val/8507119.jpg


In [11]:
df.to_parquet(
    "data/danbooru_after2024_testset.parquet",
    index=False,
    compression="zstd",
)